In [ ]:
import json
import os
import time

import dask
import dask.dataframe as dd
import numpy as np
import pandas as pd

from dask.diagnostics import ProgressBar

### Functions

In [ ]:
def getFiles(filepath):
    files = []
    for file in os.listdir(filepath):
        if not file.startswith('.'):
            # if not 'C2' in file:
            files.append(os.path.join(filepath, file))
    return files

In [ ]:
def getNHDFiles(filepath, bwidth):
    files = []
    for file in os.listdir(filepath):
        if not file.startswith('.'):
            if bwidth in file:
                files.append(os.path.join(filepath, file))
    return files

### Setup

In [ ]:
# Order of all possible bins for plotting
order = ['(0, 10]', '(10, 20]', '(20, 30]', '(30, 40]', '(40, 50]',
         '(50, 60]', '(60, 70]', '(70, 80]', '(80, 90]', '(90, 100]',
         '(100, 150]', '(150, 200]', '(200, 500]', '(500, 1000]']

date = '_2025_12_07_'

save_dir = '/nas/cee-water/cjgleason/fiona/for_manuscripts/26_narrow_rivers/results/'

## Read in and prep results, keeping overlap

### HUC2_01, Northeast

In [ ]:
## Setup
bwidth = 'mean'
bbin = 'Bin'
huc2 = '01'
output_dir = f'../../narrow_rivers_PIXC_output/PIXC_vD_HUC2_{huc2}{date}{bwidth}'

nhd_dir = f'/nas/cee-water/cjgleason/fiona/narrow_rivers_PIXC_data/NHD_prepped_buffered/HUC2_{huc2}'

In [ ]:
# Get list of files
files = getFiles(output_dir)
print(len(files))

# Read in all files
ddf_ne = dd.read_parquet(files)

In [ ]:
# Get list of NHDfiles
nhd_files = getNHDFiles(nhd_dir, bwidth)
print(len(nhd_files))

# Read in all files
nhd_ne = dd.read_parquet(nhd_files, columns=['NHDPlusID', 'GNIS_Name',
                                             'LengthKM', 'WidthM',
                                             'Bin', 'StreamOrde',
                                             'Slope', 'Sinuosity'])

In [ ]:
nhd_ne = nhd_ne.drop_duplicates(subset=['NHDPlusID'])

In [ ]:
ddf_ne = ddf_ne.drop(columns=['LengthKM', 'Slope', 'WidthM',
                              'Azimuth', 'Sinuosity', 'GNIS_Name',
                              'Bin'])

In [ ]:
ddf_ne = ddf_ne.merge(right=nhd_ne, how='left', on='NHDPlusID')

In [ ]:
# Drop all rivers < 10 m wide
ddf_ne = ddf_ne[ddf_ne['WidthM'] >= 10]

In [ ]:
# I acidentally failed to rename this column
ddf_ne = ddf_ne.rename(columns={'pixel_area': 'area_total'})

# Calculate the coverage
ddf_ne['area_eff_frac'] = ddf_ne['area_weighted'] / ddf_ne['area_total']
ddf_ne['coverage'] = ddf_ne['pseudo_area_clip'] * ddf_ne['area_eff_frac'] / ddf_ne['reach_area']

# Fill zeros so we can count them
# ddf_ne['coverage'] = ddf_ne['coverage'].fillna(0)

In [ ]:
ddf_ne['region'] = 'NE'

### HUC2_03, Southeast

In [ ]:
## Setup
bwidth = 'mean'
bbin = 'Bin'
huc2 = '03'
output_dir = f'../../narrow_rivers_PIXC_output/PIXC_vD_HUC2_{huc2}{date}{bwidth}'

nhd_dir = f'/nas/cee-water/cjgleason/fiona/narrow_rivers_PIXC_data/NHD_prepped_buffered/HUC2_{huc2}'

In [ ]:
# Get list of files
files = getFiles(output_dir)
len(files)

In [ ]:
ddf_se = dd.read_parquet(files)

In [ ]:
# Get list of NHDfiles
nhd_files = getNHDFiles(nhd_dir, bwidth)
print(len(nhd_files))

# Read in all files
nhd_se = dd.read_parquet(nhd_files, columns=['NHDPlusID', 'GNIS_Name',
                                             'LengthKM', 'WidthM',
                                             'Bin', 'StreamOrde',
                                             'Slope', 'Sinuosity'])

In [ ]:
nhd_se = nhd_se.drop_duplicates(subset=['NHDPlusID'])

In [ ]:
ddf_se = ddf_se.drop(columns=['LengthKM', 'Slope', 'WidthM',
                              'Azimuth', 'Sinuosity', 'GNIS_Name',
                              'Bin'])

In [ ]:
ddf_se = ddf_se.merge(right=nhd_se, how='left', on='NHDPlusID')

In [ ]:
# Drop all rivers < 10 m wide
ddf_se = ddf_se[ddf_se['WidthM'] >= 10]

In [ ]:
# I acidentally failed to rename this column
ddf_se = ddf_se.rename(columns={'pixel_area': 'area_total'})

# Calculate the coverage
ddf_se['area_eff_frac'] = ddf_se['area_weighted'] / ddf_se['area_total']
ddf_se['coverage'] = ddf_se['pseudo_area_clip'] * ddf_se['area_eff_frac'] / ddf_se['reach_area']

# Fill zeros so we can count them
# ddf_se['coverage'] = ddf_se['coverage'].fillna(0)

In [ ]:
ddf_se['region'] = 'SE'

### HUC2_15, Southwest

In [ ]:
## Setup
bwidth = 'mean'
bbin = 'Bin'
huc2 = '15'
output_dir = f'../../narrow_rivers_PIXC_output/PIXC_vD_HUC2_{huc2}{date}{bwidth}'

nhd_dir = f'/nas/cee-water/cjgleason/fiona/narrow_rivers_PIXC_data/NHD_prepped_buffered/HUC2_{huc2}'

In [ ]:
# Get list of files
files = getFiles(output_dir)
len(files)

In [ ]:
ddf_sw = dd.read_parquet(files)

In [ ]:
# Get list of NHDfiles
nhd_files = getNHDFiles(nhd_dir, bwidth)
print(len(nhd_files))

# Read in all files
nhd_sw = dd.read_parquet(nhd_files, columns=['NHDPlusID', 'GNIS_Name',
                                             'LengthKM', 'WidthM',
                                             'Bin', 'StreamOrde',
                                             'Slope', 'Sinuosity'])

In [ ]:
nhd_sw = nhd_sw.drop_duplicates(subset=['NHDPlusID'])

In [ ]:
ddf_sw = ddf_sw.drop(columns=['LengthKM', 'Slope', 'WidthM',
                              'Azimuth', 'Sinuosity', 'GNIS_Name',
                              'Bin'])

In [ ]:
ddf_sw = ddf_sw.merge(right=nhd_sw, how='left', on='NHDPlusID')

In [ ]:
# Drop all rivers < 10 m wide
ddf_sw = ddf_sw[ddf_sw['WidthM'] >= 10]

In [ ]:
# I acidentally failed to rename this column
ddf_sw = ddf_sw.rename(columns={'pixel_area': 'area_total'})

# Calculate the coverage
ddf_sw['area_eff_frac'] = ddf_sw['area_weighted'] / ddf_sw['area_total']
ddf_sw['coverage'] = ddf_sw['pseudo_area_clip'] * ddf_sw['area_eff_frac'] / ddf_sw['reach_area']

# Fill zeros so we can count them
# ddf_sw['coverage'] = ddf_sw['coverage'].fillna(0)

In [ ]:
ddf_sw['region'] = 'SW'

### HUC2_17, Northwest

In [ ]:
## Setup
bwidth = 'mean'
bbin = 'Bin'
huc2 = '17'
output_dir = f'../../narrow_rivers_PIXC_output/PIXC_vD_HUC2_{huc2}{date}{bwidth}'

nhd_dir = f'/nas/cee-water/cjgleason/fiona/narrow_rivers_PIXC_data/NHD_prepped_buffered/HUC2_{huc2}'

In [ ]:
# Get list of files
files = getFiles(output_dir)
len(files)

In [ ]:
ddf_nw = dd.read_parquet(files)

In [ ]:
# Get list of NHDfiles
nhd_files = getNHDFiles(nhd_dir, bwidth)
print(len(nhd_files))

# Read in all files
nhd_nw = dd.read_parquet(nhd_files, columns=['NHDPlusID', 'GNIS_Name',
                                             'LengthKM', 'WidthM',
                                             'Bin', 'StreamOrde',
                                             'Slope', 'Sinuosity'])

In [ ]:
nhd_nw = nhd_nw.drop_duplicates(subset=['NHDPlusID'])

In [ ]:
ddf_nw = ddf_nw.drop(columns=['LengthKM', 'Slope', 'WidthM',
                              'Azimuth', 'Sinuosity', 'GNIS_Name',
                              'Bin'])

In [ ]:
ddf_nw = ddf_nw.merge(right=nhd_nw, how='left', on='NHDPlusID')

In [ ]:
# Drop all rivers < 10 m wide
ddf_nw = ddf_nw[ddf_nw['WidthM'] >= 10]

In [ ]:
# I acidentally failed to rename this column
ddf_nw = ddf_nw.rename(columns={'pixel_area': 'area_total'})

# Calculate the coverage
ddf_nw['area_eff_frac'] = ddf_nw['area_weighted'] / ddf_nw['area_total']
ddf_nw['coverage'] = ddf_nw['pseudo_area_clip'] * ddf_nw['area_eff_frac'] / ddf_nw['reach_area']

# Fill zeros so we can count them
# ddf_nw['coverage'] = ddf_nw['coverage'].fillna(0)

In [ ]:
ddf_nw['region'] = 'NW'

### Combine HUCs

In [ ]:
ddf = dd.concat([ddf_ne, ddf_se, ddf_sw, ddf_nw])
ddf = ddf[['NHDPlusID', 'num_wse', 'coverage', 'Bin', 'region']]
ddf = ddf.repartition(npartitions=15) 

### Perenniality

In [ ]:
from pathlib import Path

data_dir = Path('/nas/cee-water/cjgleason/fiona/data/CONUS_ephemeral_results_v2/results_archive/')
files = sorted(data_dir.glob('*.csv'))

ephem = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)

ephem['NHDPlusID'] = ephem['NHDPlusID'].astype('float64')
ephem = ephem[['NHDPlusID', 'perenniality']].drop_duplicates()

In [ ]:
ephem_dd = dd.from_pandas(ephem, npartitions=1)

In [ ]:
ddf = ddf.merge(ephem_dd, on='NHDPlusID', how='left')

## Calc stats

In [ ]:
def computeStats(ddf, bbin):
    # CRITICAL FIX: Cast the grouping column to string to strip latent categorical metadata
    # This prevents Dask from tracking unobserved category levels across partitions
    ddf = ddf.assign(**{bbin: ddf[bbin].astype(str)})

    # 1. Capture ALL unique reaches per bin before applying any filters
    all_reaches_task = (
        ddf.groupby([bbin, 'NHDPlusID'])
        .size()
        .to_frame(name='total_raw_observations')
        .reset_index()
    )
    
    total_bin_stats_task = (
        all_reaches_task.groupby(bbin)
        .agg({'NHDPlusID': 'count'})
        .rename(columns={'NHDPlusID': 'total_unique_reaches'})
    )
    
    # 2. Filter for rows with at least 5 pixels
    ddf_filtered = ddf[ddf['num_wse'] >= 5]
    
    # Track the reach counts cleanly
    reach_counts_task = (
        ddf_filtered.groupby([bbin, 'NHDPlusID'])
        .size()
        .to_frame(name='observations')
        .reset_index()
    )
    
    # 3. Pull intermediate stats down to local Pandas DataFrames
    print("Computing baseline total reaches per bin...")
    with ProgressBar():
        total_stats = total_bin_stats_task.compute()
        
    print("Computing raw filtered reach counts...")
    with ProgressBar():
        reach_counts = reach_counts_task.compute()
        
    # 4. Perform final logical checks safely in Pandas
    print("Evaluating frequency metrics and unifying final datasets...")
    reach_counts['is_frequent'] = (reach_counts['observations'] >= 20).astype(int)
    
    filtered_stats = (
        reach_counts.groupby(bbin)
        .agg({
            'NHDPlusID': 'count',
            'is_frequent': 'sum'
        })
        .rename(columns={
            'NHDPlusID': 'reaches_with_valid_obs',
            'is_frequent': 'reaches_gte_20_count'
        })
    )
    
    # 5. Perform the final pandas merge
    stats = pd.merge(total_stats, filtered_stats, on=bbin, how='left')
    
    # 6. Clean up, sort, handle NaNs from the left join, and calculate percentages
    stats = stats.reset_index().sort_values(by=bbin).reset_index(drop=True)
    
    stats['reaches_with_valid_obs'] = stats['reaches_with_valid_obs'].fillna(0).astype(int)
    stats['reaches_gte_20_count'] = stats['reaches_gte_20_count'].fillna(0).astype(int)

    stats['pct_reaches_valid_obs'] = ((stats['reaches_with_valid_obs'] / stats['total_unique_reaches']) * 100).round(2)
    stats['pct_reaches_gte_20'] = ((stats['reaches_gte_20_count'] / stats['total_unique_reaches']) * 100).round(2)

    stats['pct_reaches_total'] = 100
    
    return stats

In [ ]:
stats = computeStats(ddf=ddf, bbin=bbin)

stats['Bin'] = stats['Bin'].astype(str)

In [ ]:
# stats = computeStats(ddf=ddf, bbin=bbin)
# stats['Bin'] = stats['Bin'].astype(str)

stats.to_parquet(os.path.join(save_dir, 'area_observed_counts.parquet'))

#### By region

In [ ]:
# Quick test to look at the raw distribution
ddf_filtered = ddf_ne[ddf_ne['num_wse'] >= 5]
reach_counts = ddf_filtered.groupby(['Bin', 'NHDPlusID']).size().to_frame(name='observations').reset_index().compute()

print("--- Data Distribution Diagnostic ---")
print(f"Total rows in reach_counts: {len(reach_counts)}")
print(f"Maximum observations found on any single reach: {reach_counts['observations'].max()}")
print("\nTop 5 highest observation counts:")
print(reach_counts['observations'].nlargest(5))

In [ ]:
# NE
# ddf_ne = ddf_ne[['NHDPlusID', 'num_wse', 'coverage', 'Bin']]
# ddf_ne = ddf_ne.repartition(npartitions=15)
stats_ne = computeStats(ddf=ddf[ddf['region'] == 'NE'], bbin=bbin)

stats_ne['Bin'] = stats_ne['Bin'].astype(str)

stats_ne.to_parquet(os.path.join(save_dir, 'area_observed_counts_ne.parquet'))

In [ ]:
# SE
# ddf_se = ddf_se.repartition(npartitions=15)
stats_se = computeStats(ddf=ddf[ddf['region'] == 'SE'], bbin=bbin)

stats_se['Bin'] = stats_se['Bin'].astype(str)

stats_se.to_parquet(os.path.join(save_dir, 'area_observed_counts_se.parquet'))

In [ ]:
# SW
# ddf_sw = ddf_sw.repartition(npartitions=15)
stats_sw = computeStats(ddf=ddf[ddf['region'] == 'SW'], bbin=bbin)

stats_sw['Bin'] = stats_sw['Bin'].astype(str)

stats_sw.to_parquet(os.path.join(save_dir, 'area_observed_counts_sw.parquet'))

In [ ]:
# NW
# ddf_nw = ddf_nw.repartition(npartitions=15)
stats_nw = computeStats(ddf=ddf[ddf['region'] == 'NW'], bbin=bbin)

stats_nw['Bin'] = stats_nw['Bin'].astype(str)

stats_nw.to_parquet(os.path.join(save_dir, 'area_observed_counts_nw.parquet'))